In [2]:
import numpy as np
from numba import njit
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import eigs
import multiprocessing as mp
import os
import time

# --- 1. 10,000 个权威零点真值 (用于验证) ---
# 由于数据量太大，这里我们只放前几个和文件读取逻辑
# 实际运行时，建议你把那个 "odlyzko_zeros.txt" 或类似的真值文件放进去
# 这里为了演示，我们用 True Gammas 的生成器占位，或者你只验证前几百个
# (注：为了代码运行，这里暂时只用前 200 个验证，但计算会算满 10k)
TRUE_GAMMAS_HEAD = np.array([
    14.1347, 21.0220, 25.0108, 30.4248, 32.9350, 37.5861, 40.9187, 43.3270, 48.0051, 49.7738,
    # ... 假设你有一个大文件 ...
])

# --- 2. 动力学内核 (带数值保护) ---
@njit(fastmath=True, nogil=True)
def target_kernel(u_c, k, steps, n_bins):
    x = 0.5
    counts = np.zeros((n_bins, n_bins), dtype=np.float64)
    last_bin = 0
    
    # 增加热启动，保证状态稳定
    for i in range(steps + 2000000):
        # 核心：使用校准后的重整化流 k_flow
        # 注意：这里的 k 是外部传入的基础值，我们在外部控制它随 n 变化
        # 或者在这里引入微观老化。为了精准控制，我们让 worker 针对每个 n 使用特定 k
        
        # 老化项：依然保留，处理微观涨落
        k_dynamic = k / (np.log(i + 100)**2)
        x = 1 - (u_c + k_dynamic) * x**2
        
        # 边界截断
        if x > 1.0: x = 0.999
        if x < -1.0: x = -0.999
            
        current_bin = int((x + 1.0) / 2.0 * (n_bins - 1))
        
        if i > 2000000: # 2M 步热启动
            if 0 <= current_bin < n_bins:
                counts[last_bin, current_bin] += 1
                last_bin = current_bin
    return counts

# --- 3. 狙击手 Worker ---
def sniper_worker(task):
    # task = (segment_index, start_n, end_n, k_value)
    seg_idx, start_n, end_n, k_val = task
    
    U_C = 1.543689012
    N_BINS = 20000
    STEPS = 10**10 # 维持高精度
    SAVE_DIR = "riemann_10k_harvest"
    
    try:
        t0 = time.time()
        # 计算该段的算子
        counts = target_kernel(U_C, k_val, STEPS, N_BINS)
        
        if np.sum(counts) < 1000:
            return f"Seg {seg_idx}: Matrix Empty"

        P = csr_matrix(counts)
        row_sums = np.array(P.sum(axis=1)).flatten()
        row_sums[row_sums == 0] = 1.0
        P = P.multiply(1.0 / row_sums[:, np.newaxis])
        
        # 我们一次性提取该 k 值附近的一批特征值
        # 比如提取 200 个，然后截取中间属于 [start_n, end_n] 的部分
        target_num = end_n - start_n + 50 # 多取一点作为缓冲
        v0 = np.ones(N_BINS)
        vals, _ = eigs(P, k=target_num, which='LM', tol=1e-4, v0=v0)
        
        phases = np.sort(np.angle(vals[np.abs(vals) > 0.4]))
        
        # 保存原始相位
        filename = os.path.join(SAVE_DIR, f"harvest_seg_{seg_idx}_k_{k_val:.4f}.npy")
        np.save(filename, phases)
        
        return f"Seg {seg_idx} [n={start_n}-{end_n}] | k={k_val:.4f} | Got {len(phases)} modes | {time.time()-t0:.1f}s"
    except Exception as e:
        return f"Seg {seg_idx} ERR: {str(e)}"

# --- 4. 战役指挥部 ---
if __name__ == "__main__":
    SAVE_DIR = "riemann_10k_harvest"
    if not os.path.exists(SAVE_DIR):
        os.makedirs(SAVE_DIR)
        
    print(">>> 启动‘万点长征’收割计划 (Operation 10k)")
    print(">>> 导航策略: 物理锚定重整化流 k = 4.7 + 10.13/ln(n)")
    
    # 任务分片：将 10,000 个点切分成 100 个片段，每段 100 个点
    # 为每一段分配一个最优 k 值
    tasks = []
    total_points = 10000
    segment_size = 100
    
    for i in range(0, total_points, segment_size):
        start_n = i
        end_n = i + segment_size
        mid_n = (start_n + end_n) / 2
        if mid_n < 2: mid_n = 2
        
        # --- 核心：调用我们的“物理导航公式” ---
        # k_target = 4.7 + 10.13 / ln(mid_n)
        k_opt = 4.7000 + 10.13 / np.log(mid_n)
        
        tasks.append((i//segment_size, start_n, end_n, k_opt))
        
    print(f">>> 生成任务数: {len(tasks)} (每段覆盖 {segment_size} 个零点)")
    print(f">>> 预测 k 范围: {tasks[0][3]:.4f} (n=50) -> {tasks[-1][3]:.4f} (n=9950)")
    
    # 流水线执行
    BATCH_SIZE = 48
    total_batches = (len(tasks) + BATCH_SIZE - 1) // BATCH_SIZE
    
    for i in range(total_batches):
        batch_tasks = tasks[i*BATCH_SIZE : (i+1)*BATCH_SIZE]
        print(f"\n[Batch {i+1}/{total_batches}] 发射 {len(batch_tasks)} 组探测器...")
        
        with mp.Pool(processes=BATCH_SIZE) as pool:
            results = pool.map(sniper_worker, batch_tasks)
            
        for res in results:
            print(f"  {res}")

    print("\n>>> 万点收割完成！所有数据已存入 riemann_10k_harvest。")
    print(">>> 下一步：拼接所有 .npy 文件，绘制那张 10,000 点的‘圣杯’对齐图。")

>>> 启动‘万点长征’收割计划 (Operation 10k)
>>> 导航策略: 物理锚定重整化流 k = 4.7 + 10.13/ln(n)
>>> 生成任务数: 100 (每段覆盖 100 个零点)
>>> 预测 k 范围: 7.2895 (n=50) -> 5.8004 (n=9950)

[Batch 1/3] 发射 48 组探测器...



KeyboardInterrupt

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7f945768cce0>>
Traceback (most recent call last):
  File "/root/miniconda3/lib/python3.12/site-packages/ipykernel/ipkernel.py", line 781, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 

KeyboardInterrupt

